# PokéAgent V2 PPO on Colab T4
This notebook keeps restricted files and checkpoints in Google Drive. Select a T4 GPU runtime before running.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import subprocess
from pathlib import Path

repo = Path('/content/pokemonTCG')
subprocess.run(['nvidia-smi'], check=True)
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(repo), 'switch', 'colab'], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(
        [
            'git', 'clone', '--branch', 'colab',
            'https://github.com/yqqxyy/pokemonTCG.git', str(repo),
        ],
        check=True,
    )
%cd /content/pokemonTCG
!python -m pip install -q -e . --no-deps

In [ ]:
from pathlib import Path

drive_root = Path('/content/drive/MyDrive/pokemonTCG')
archive = drive_root / 'pokemon-tcg-ai-battle.zip'
input_checkpoint = drive_root / 'checkpoints/bc_rule_v2_transformer_2000.pt'
output_checkpoint = drive_root / 'checkpoints/ppo_v2_colab.pt'
assert archive.is_file(), archive
assert input_checkpoint.is_file(), input_checkpoint
output_checkpoint.parent.mkdir(parents=True, exist_ok=True)
print(archive, input_checkpoint, output_checkpoint, sep='\n')

In [ ]:
import subprocess

import torch

from poketcg.engine import OfficialEngine

official_dir = Path('data/official')
official_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(
    ['unzip', '-q', '-o', str(archive), 'sample_submission/*', '-d', str(official_dir)],
    check=True,
)
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab'
print(torch.__version__, torch.cuda.get_device_name(0))
print(OfficialEngine().sample_deck_path)

In [ ]:
!bash scripts/train_colab_ppo.sh "{input_checkpoint}" "{output_checkpoint}"

In [ ]:
!sha256sum "{output_checkpoint}"
print('Saved:', output_checkpoint)